In [1]:
!pip install torch transformers sentencepiece datasets faiss-cpu streamlit langdetect

  Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
  Using cached datasets-4.5.0-py3-none-any.whl (515 kB)
  Using cached faiss_cpu-1.13.0-cp39-cp39-win_amd64.whl (18.7 MB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
  Using cached multiprocess-0.70.18-py39-none-any.whl (133 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl (200 kB)
  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
  Using cached aiohttp-3.13.3-cp39-cp39-win_amd64.whl (457 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
  Using cached dill-0.4.0-py3-none-any.whl (119 kB)
  Using cached frozenlist-1.8.0-cp39-cp39-win_amd64.whl (44 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl (6.2 kB)
  Using cached multidict-6.7.1-cp39-cp39-win_amd64.whl (46 kB)
  Using cached yarl-1.22.0-cp39-cp


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.8.0+cpu
CUDA available: False


In [3]:
import pandas as pd

data = pd.read_csv("agriculture_dataset.csv")

print(data.head())
print("Columns:", data.columns)
print("Total rows:", len(data))

                                            question  \
0         why is crop rotation important in farming?   
1  What farming practice helps prevent soil erosion?   
2                              what is crop rotation   
3      what are the different methods of irrigation?   
4                          why is soil health vital?   

                                             answers  
0  This helps to prevent soil erosion and depleti...  
1                                      Crop Rotation  
2  Crop rotation is the practice of growing a ser...  
3  surface irrigation, drip irrigation, and sprin...  
4  Soil health is critical to crop growth and pro...  
Columns: Index(['question', 'answers'], dtype='object')
Total rows: 22615


In [4]:
import pandas as pd

data = pd.read_csv("agriculture_dataset.csv")

print(data.columns)

Index(['question', 'answers'], dtype='object')


In [5]:
data["question"]
data["answers"]

0        This helps to prevent soil erosion and depleti...
1                                            Crop Rotation
2        Crop rotation is the practice of growing a ser...
3        surface irrigation, drip irrigation, and sprin...
4        Soil health is critical to crop growth and pro...
                               ...                        
22610    Yes, it makes them small and not good for plan...
22611             The pathogen of Phoma Rot is seed-borne.
22612                   Yes, but not as much as sugar cane
22613                   No, it does not affect germination
22614                                                   No
Name: answers, Length: 22615, dtype: object

In [6]:
print(data.columns)

Index(['question', 'answers'], dtype='object')


In [9]:
import pandas as pd
import re

data = pd.read_csv("agriculture_dataset.csv")

text = " ".join(data["question"].astype(str))

tokens = re.findall(r"\w+|[^\w\s]", text)

print(tokens[:30])
print("Total tokens:", len(tokens))

['why', 'is', 'crop', 'rotation', 'important', 'in', 'farming', '?', 'What', 'farming', 'practice', 'helps', 'prevent', 'soil', 'erosion', '?', 'what', 'is', 'crop', 'rotation', 'what', 'are', 'the', 'different', 'methods', 'of', 'irrigation', '?', 'why', 'is']
Total tokens: 223136


In [10]:
print(data.head())

                                            question  \
0         why is crop rotation important in farming?   
1  What farming practice helps prevent soil erosion?   
2                              what is crop rotation   
3      what are the different methods of irrigation?   
4                          why is soil health vital?   

                                             answers  
0  This helps to prevent soil erosion and depleti...  
1                                      Crop Rotation  
2  Crop rotation is the practice of growing a ser...  
3  surface irrigation, drip irrigation, and sprin...  
4  Soil health is critical to crop growth and pro...  


In [11]:
vocab = sorted(set(tokens))
vocab_size = len(vocab)

word_to_id = {word:i for i,word in enumerate(vocab)}
id_to_word = {i:word for word,i in word_to_id.items()}

print("Vocabulary size:", vocab_size)

Vocabulary size: 2403


In [12]:
encoded = [word_to_id[word] for word in tokens]

print(encoded[:30])

[2364, 1349, 798, 1932, 1268, 1274, 1044, 40, 396, 1044, 1740, 1226, 1761, 2036, 994, 40, 2352, 1349, 798, 1932, 2352, 518, 2195, 877, 1500, 1604, 1348, 40, 2364, 1349]


In [13]:
from torch.utils.data import Dataset

class GPTDataset(Dataset):

    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]

        return torch.tensor(x), torch.tensor(y)

In [14]:
import torch
import torch.nn as nn

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_size=128, num_heads=4, num_layers=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=embed_size,
                nhead=num_heads
            ),
            num_layers=num_layers
        )

        self.fc = nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = self.fc(x)
        return x

In [15]:
model = MiniGPT(vocab_size)
print(model)

MiniGPT(
  (embedding): Embedding(2403, 128)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=128, out_features=2403, bias=True)
)


C:\Users\Dell\anaconda3\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [16]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [17]:
import torch
from torch.utils.data import Dataset, DataLoader

In [18]:
import torch
from torch.utils.data import Dataset

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):

            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [19]:
import torch
from torch.utils.data import Dataset

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [20]:
with open("agriculture_text.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(raw_text[:200])

Agriculture is the practice of cultivating soil, growing crops, and raising livestock for food and other products.

Wheat is one of the most important cereal crops in the world. Wheat grows best in we


In [21]:
!pip install tiktoken


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [23]:
dataset = GPTDatasetV1(
    txt=raw_text,
    tokenizer=tokenizer,
    max_length=64,
    stride=32
)

In [24]:
from torch.utils.data import DataLoader

In [25]:
dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

In [26]:
for x, y in dataloader:
    print(x.shape, y.shape)
    break

torch.Size([4, 64]) torch.Size([4, 64])


In [27]:
vocab_size = tokenizer.n_vocab
print(vocab_size)

50257


In [28]:
import torch
from torch.utils.data import Dataset

class GPTDatasetV1(Dataset):

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):

            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [29]:
with open("agriculture_text.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(raw_text[:200])
print("Characters:", len(raw_text))

Agriculture is the practice of cultivating soil, growing crops, and raising livestock for food and other products.

Wheat is one of the most important cereal crops in the world. Wheat grows best in we
Characters: 1860


In [30]:
dataset = GPTDatasetV1(
    txt=raw_text,
    tokenizer=tokenizer,
    max_length=64,
    stride=32
)

In [31]:
import torch
import torch.nn as nn

vocab_size = 50257
embed_dim = 128

class SimpleGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4),
            num_layers=2
        )

        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)   # ← self is valid here
        x = self.transformer(x)
        x = self.fc(x)
        return x


model = SimpleGPT()

C:\Users\Dell\anaconda3\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [32]:
model = SimpleGPT()

In [33]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [34]:
epochs = 5

for epoch in range(epochs):
    for x, y in dataloader:
        optimizer.zero_grad()

        output = model(x)

        loss = criterion(output.view(-1, vocab_size), y.view(-1))

        loss.backward()
        optimizer.step()

    print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 9.852852821350098
Epoch: 1 Loss: 8.928150177001953
Epoch: 2 Loss: 8.07611083984375
Epoch: 3 Loss: 7.420226573944092
Epoch: 4 Loss: 6.83384370803833


In [35]:
!pip install langdetect


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from langdetect import detect

text = "कापूस पिकासाठी कोणते खत वापरावे?"

language = detect(text)

print(language)

mr


In [37]:
!pip install streamlit


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import streamlit as st

st.title("Smart Agriculture AI Assistant 🌾")

question = st.text_input("Ask your farming question")

if question:

    answer = "Example: Use nitrogen fertilizer for wheat."

    st.write("AI Advice:", answer)

2026-03-10 20:35:14.481 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-10 20:35:14.767 
  command:

    streamlit run C:\Users\Dell\anaconda3\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-10 20:35:14.767 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-10 20:35:14.770 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-10 20:35:14.771 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-10 20:35:14.772 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-10 20:35:14.773 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-10 20:35:14.775 Thread 'MainThread': mis

In [42]:
print(data.head())
print("Total rows:", len(data))

                                            question  \
0         why is crop rotation important in farming?   
1  What farming practice helps prevent soil erosion?   
2                              what is crop rotation   
3      what are the different methods of irrigation?   
4                          why is soil health vital?   

                                             answers  
0  This helps to prevent soil erosion and depleti...  
1                                      Crop Rotation  
2  Crop rotation is the practice of growing a ser...  
3  surface irrigation, drip irrigation, and sprin...  
4  Soil health is critical to crop growth and pro...  
Total rows: 22615


In [43]:
import pandas as pd

data = pd.read_csv("agriculture_dataset.csv", on_bad_lines="skip")

cotton_rows = data[data["question"].str.contains("cotton", case=False, na=False)]

print("Total cotton questions:", len(cotton_rows))
print(cotton_rows.head())

Total cotton questions: 2
                                 question  \
22616     How to control pests in cotton?   
22617  How to prevent bollworm in cotton?   

                                                 answers  
22616  Use neem oil spray or recommended pesticides l...  
22617  Use pheromone traps and recommended insecticid...  


In [44]:
pip install sentence-transformers faiss-cpu

     -------------------------------------- 488.0/488.0 kB 6.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
